In [1]:
%load_ext autoreload
%autoreload 2


import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchvision.utils as vutils
import sklearn
from PIL import Image

from compcats.datasets.fmri import nsd
from compcats.utils import model_utils, responsecomparison_utils as rc_utils
from compcats.models import  resnet50_openaiclip

RESULTS_DIR = 'results/'
os.makedirs(RESULTS_DIR, exist_ok=True)
PLOTS_DIR = 'plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

NSD_DIR = '/weka/scratch/weka/sinha/alishd/datasets/nsd'
MODELS_DIR = '/om2/user/alishd/github/comparing-category-selectivity/preprint/models'

In [2]:
# Utils
# Get end to end torch encoding model
class EM(torch.nn.Module):
    def __init__(self, model, layer, mapping_coef, mapping_intercept):
        super(EM, self).__init__()
        self.model = model
        self.layer = layer
        
        # Convert sklearn mapping to torch.nn.Linear
        n_features = mapping_coef.shape[1]
        n_outputs = mapping_coef.shape[0]
        self.mapping_layer = torch.nn.Linear(n_features, n_outputs)
        
        # Set weights and bias from the sklearn model
        self.mapping_layer.weight.data = torch.FloatTensor(mapping_coef)
        self.mapping_layer.bias.data = torch.FloatTensor(mapping_intercept)
        
        # Get the layer module
        self.layer_module = dict([*self.model.model.named_modules()])[self.layer]
        
        # Register a forward hook to get the activations
        self.activations = None
        def hook(module, input, output):
            self.activations = output
        self.layer_module.register_forward_hook(hook)
    
    def forward(self, x):
        _ = self.model(x)
        # Get the activations
        x = self.activations
        x = x.reshape(x.shape[0], -1)
        # Apply the mapping directly with torch Linear layer
        x_mapped = self.mapping_layer(x)
        return x_mapped

In [3]:
subjects = [1, 2, 5, 7]
tthreshold_dict = {
    'rOFA': 3,
    'rFFA': 6
}
frois = ['rFFA', 'rOFA']

train_subset = 485
test_subset = 515

model = resnet50_openaiclip.get_model()
layer = 'layer4.1.conv3'
alpha = 0.1
seed = 42

In [6]:
resultd_save_dir = f'{RESULTS_DIR}/EMs_nolayersweep/'
os.makedirs(resultd_save_dir, exist_ok=True)

all_results = {}
for froi in frois:
    all_results[froi] = {}
    fmri_tthreshold = tthreshold_dict[froi]
    for subject in subjects:
        # Get subject data
        Y_trainval, _ = nsd.get_betas(
            NSD_DIR, subject, froi, fmri_tthreshold, subset=train_subset, voxels=True
        )
        Y_trainval = Y_trainval.cpu().numpy()
        Y_test, _ = nsd.get_betas(
            NSD_DIR, subject, froi, fmri_tthreshold, subset=test_subset, voxels=True
        )
        # Get activations for the model and layer
        X_trainval_fulllayer = model_utils.get_dataset_activations(
            MODELS_DIR, NSD_DIR, model, layer, 'nsd', subset=train_subset
        )
        X_test_fulllayer = model_utils.get_dataset_activations(
            MODELS_DIR, NSD_DIR, model, layer,
            'nsd', subset=test_subset
        )
        # Layer
        X_trainval = X_trainval_fulllayer.reshape(X_trainval_fulllayer.shape[0], -1).cpu().numpy()
        X_test = X_test_fulllayer.reshape(X_test_fulllayer.shape[0], -1).cpu().numpy()
        # Map
        print(f'FROI: {froi}, Model: {model.name}, Subject: {subject}, tthreshold: {fmri_tthreshold}, mapper: Ridge')
        mapping = mapping = sklearn.linear_model.Ridge(alpha=alpha, random_state=seed)
        # Fit the mapping
        mapping.fit(X_trainval, Y_trainval)
        # Get train performance
        Y_trainval = torch.FloatTensor(Y_trainval)
        y_pred_trainval = torch.FloatTensor(mapping.predict(X_trainval))
        train_pearsonr = rc_utils.pearsonr(Y_trainval.mean(axis=-1), y_pred_trainval.mean(axis=-1))
        train_rdmr = rc_utils.rdmr(Y_trainval, y_pred_trainval)
        print(f'Train Pearsonr: {train_pearsonr:.4f}, Train RDMR: {train_rdmr:.4f}')
        # Get test performance
        y_pred_test = torch.FloatTensor(mapping.predict(X_test))
        test_pearsonr = rc_utils.pearsonr(Y_test.mean(axis=-1), y_pred_test.mean(axis=-1))
        test_rdmr = rc_utils.rdmr(Y_test, y_pred_test)
        print(f'Test Pearsonr: {test_pearsonr:.4f}, Test RDMR: {test_rdmr:.4f}')

        # Save results
        results = {
            'froi': froi,
            'model': model.name,
            'layer': layer,
            'subject': subject,
            'tthreshold': tthreshold_dict[froi],
            'mapper': 'Ridge',
            'alpha': alpha,
            'train_pearsonr': train_pearsonr,
            'train_rdmr': train_rdmr,
            'test_pearsonr': test_pearsonr,
            'test_rdmr': test_rdmr,
            'coef_': mapping.coef_,
            'intercept_': mapping.intercept_,
        }
        all_results[froi][subject] = results
results_file_name = f'EM_nolayersweep_{model.name}_{layer}_ridge_alpha{alpha}.npz'
np.savez(
    os.path.join(RESULTS_DIR, results_file_name),
    **all_results
)

FROI: rFFA, Model: resnet50openaiclip, Subject: 1, tthreshold: 6, mapper: Ridge
Train Pearsonr: 1.0000, Train RDMR: 1.0000
Test Pearsonr: 0.7712, Test RDMR: 0.5452
FROI: rFFA, Model: resnet50openaiclip, Subject: 2, tthreshold: 6, mapper: Ridge
Train Pearsonr: 1.0000, Train RDMR: 1.0000
Test Pearsonr: 0.7603, Test RDMR: 0.5193
FROI: rFFA, Model: resnet50openaiclip, Subject: 5, tthreshold: 6, mapper: Ridge
Train Pearsonr: 1.0000, Train RDMR: 1.0000
Test Pearsonr: 0.6882, Test RDMR: 0.4120
FROI: rFFA, Model: resnet50openaiclip, Subject: 7, tthreshold: 6, mapper: Ridge
Train Pearsonr: 1.0000, Train RDMR: 1.0000
Test Pearsonr: 0.7140, Test RDMR: 0.4606
FROI: rOFA, Model: resnet50openaiclip, Subject: 1, tthreshold: 3, mapper: Ridge
Train Pearsonr: 1.0000, Train RDMR: 1.0000
Test Pearsonr: 0.4743, Test RDMR: 0.1754
FROI: rOFA, Model: resnet50openaiclip, Subject: 2, tthreshold: 3, mapper: Ridge
Train Pearsonr: 1.0000, Train RDMR: 1.0000
Test Pearsonr: 0.4817, Test RDMR: 0.1531
FROI: rOFA, Mode

In [8]:
from torch.utils.data import Dataset, DataLoader

class ImageDataset(Dataset):
    """Dataset for a single folder of images, without labels."""
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        # List all files and ensure they are images (e.g., .bmp, .png, .jpg)
        self.img_names = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.bmp', '.png', '.jpg', '.jpeg'))])

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_names[idx])
        image = Image.open(img_path).convert("RGB")
        
        # We'll return the original filename along with the image.
        # This is very useful for inference tasks.
        filename = self.img_names[idx]

        if self.transform:
            image = self.transform(image)
            
        return image, filename

In [9]:
STIMULI_DIR = f'stimuli_MD'

dataset = ImageDataset(STIMULI_DIR, transform=model.transforms)

data_loader = DataLoader(
        dataset=dataset,
        batch_size=512,
        shuffle=False,      # IMPORTANT: Set to False for sequential processing
        num_workers=16
    )

In [30]:
predictions_dir = f'{RESULTS_DIR}/predictions/'
os.makedirs(predictions_dir, exist_ok=True)

for froi in frois:
    for subject in subjects:
        mapping = all_results[froi][subject]
        mapping_coef = mapping['coef_']
        mapping_intercept = mapping['intercept_']
        em = EM(model, layer, mapping_coef, mapping_intercept)
        em = em.cuda()
        em.eval()
        with torch.no_grad():
            assert len(data_loader) == 1, "DataLoader should have only one batch for all images."
            for batch in data_loader:
                images, filenames = batch
                images = images.cuda()
                preds = em(images)
        # Convert filenames and preds to a csv with first column as filenames and rest as preds
        preds = preds.cpu().numpy()
        df = pd.DataFrame(preds)
        df.insert(0, 'filename', filenames)
        output_csv = f'{predictions_dir}/predictions_{froi}_t{tthreshold_dict[froi]}_{subject}.csv'
        df.to_csv(output_csv, index=False)
        print(f'Saved predictions to {output_csv}')

torch.Size([420, 123])
torch.Size([420, 177])
torch.Size([420, 194])
torch.Size([420, 56])
torch.Size([420, 64])
torch.Size([420, 76])
torch.Size([420, 188])
torch.Size([420, 96])
